# ViewX v0.3.0 — Guia de uso

Libreria de visualizacion: **vx.plot()**, **Dashboard**, **DataMatrix**, **Slides** y **Report (PDF)**.

Todos los motores exportan con `.save(path)` y `.show()`. Un mismo tema funciona en todos (`vx.set_theme(...)`).


## 1. Introduccion e instalacion

In [15]:
import viewx as vx
vx.welcome()
# pip install viewx

ViewX v0.2.4
Librería de visualizacion
Autor: Emmanuel Ascendra

Clases disponibles:
 - HTML
 - DataMatrix
 - Report
 - Slides

Para más información: help(viewx)

O lee la información en: https://viewx.vercel.app/


## 2. Imports

In [16]:
from viewx import Dashboard, DataMatrix, Report, Presentation, Slide, load_dataset
from viewx.datasets import load_iris, load_penguins, generate_dataset
from viewx.Slides import Title, BarPlot, BulletList
import pandas as pd


## 3. Datasets

In [17]:
df = load_iris()
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [18]:
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X, y = load_iris(return_X_y=(feature_cols, "species"))
print(X.shape, y.shape)


(150, 4) (150,)


In [19]:
synthetic = generate_dataset(100, {
    "region": {"dist": "categorical", "labels": ["North", "South", "East"], "probs": [0.4, 0.35, 0.25]},
    "revenue": {"dist": "lognormal", "mean": 7, "sigma": 0.4, "round": 2},  # mean en escala log
    "units": {"dist": "poisson", "lam": 12},
})
synthetic.head()


,region,revenue,units
0,North,inf,15.0
1,East,inf,13.0
2,South,inf,8.0
3,South,inf,10.0
4,North,inf,13.0


## 4. Graficas rapidas — `vx.plot()`

Una llamada para 14 tipos de grafica. Devuelve un `Chart` con `.save()`, `.show()` y `.fig`.
Si no pasas `kind`, se infiere de los dtypes. Con `static=True` genera matplotlib (para papers/PDF).
Series de mas de 10k puntos se reducen automaticamente y los scatter grandes usan WebGL.

In [ ]:
# Interactiva (Plotly) — se muestra inline en el notebook
vx.plot(df, x="sepal_length", y="petal_length", z="species", title="Iris scatter")

# Kind inferido: datetime -> line, num-num -> scatter, cat-num -> bar, solo num -> histogram
chart = vx.plot(df, kind="box", x="species", y="petal_length", theme="void_indigo")
chart.save("output/notebook_box.html")

# Estatica (matplotlib) para papers / PDFs
vx.plot(df, kind="histogram", x="sepal_length", static=True, title="Histograma").save("output/notebook_hist.png")

print("Graficas guardadas en output/")

## 5. DataMatrix — EDA interactivo

Reporte HTML con pestanas Overview, Variables, **Explore** (filtros + graficos), Correlations, Bibliometrics y Sample. En datasets grandes el sample se limita (`sample_rows`) y el explorador usa muestreo estratificado.


In [20]:
dm = DataMatrix(df)
dm.analyze()
print(dm.summary())
print("Alerts:", dm.alerts()[:3])
print("Highlights:", dm.highlights()[:3])


{'rows': 150, 'columns': 5, 'duplicates': 3, 'missing_cells': np.int64(0), 'missing_pct': np.float64(0.0), 'memory': '13.5 KB', 'numeric': 4, 'categorical': 1, 'datetime': 0, 'boolean': 0}
Alerts: ["Column 'sepal_width': 4 outliers detected", 'Found 3 duplicate rows']
Highlights: ['Low global missing rate (0.0%)', 'Diverse column mix: 4 numeric, 1 categorical, 0 datetime, 0 boolean', 'Strong correlation: petal_length vs petal_width (r=0.96)']


In [21]:
dm.clean_data(drop_duplicates=True, fill_na=True)
dm.save("output/notebook_datamatrix.html", title="Iris EDA")  # dm.show() abre el navegador


  removed 3 duplicate rows
  filled missing values


'output/notebook_datamatrix.html'

## 6. Dashboard HTML

Constructor manual + `Dashboard.auto()` con layouts preset. `auto()` devuelve el dashboard; se exporta con `.save()`.


In [22]:
Dashboard.auto(df, title="Iris Auto Dashboard", layout="kpi_focus", theme="void_indigo").save("output/notebook_html_auto.html")


c:\Users\Usuario\Documents\Emmanuel Ascendra\GitHub_GhostAnalyst\Librerias_Python\Libreria_Visual\viewx\HTML\html_engine.py:1182: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



'output/notebook_html_auto.html'

In [23]:
dash = Dashboard(title="Iris Manual", theme="dark_enterprise", cols=12, rows=8)
dash.add_valuebox("Filas", len(df), icon_key="database", row=1, col=1, width=3, height=2)
dash.add_infobox(df, "species", row=1, col=4, width=4, height=2)
dash.add_chart(vx.plot(df, kind="scatter", x="sepal_length", y="petal_length", z="species"),
               title="Sepal vs Petal", row=1, col=8, width=5, height=6)
dash.add_table(df.head(10), title="Muestra", row=3, col=1, width=7, height=4)
dash.save("output/notebook_html_manual.html")


'output/notebook_html_manual.html'

## 7. Slides

Presentaciones manuales y automaticas desde DataFrame. `Presentation.auto()` devuelve la presentacion; se exporta con `.save()`.


In [24]:
pres = Presentation("ViewX Slides Demo", theme="dark_enterprise")
with Slide(title="Portada"):
    Title("ViewX Slides").center("x").pos(top=20).zoom_in()
    BulletList(["Componentes Plotly", "Animaciones CSS", "Temas integrados"]).pos(left=8, top=40).size(width="50%")
pres.save("output/notebook_slides_manual.html")


'c:\\Users\\Usuario\\Documents\\Emmanuel Ascendra\\GitHub_GhostAnalyst\\Librerias_Python\\Libreria_Visual\\output\\notebook_slides_manual.html'

In [25]:
Presentation.auto(df, title="Iris Auto Slides", theme="glass_ocean").save("output/notebook_slides_auto.html")


'c:\\Users\\Usuario\\Documents\\Emmanuel Ascendra\\GitHub_GhostAnalyst\\Librerias_Python\\Libreria_Visual\\output\\notebook_slides_auto.html'

## 8. Report PDF

Reporte manual y automatico con advertencias y fortalezas. Requiere **pdflatex** instalado (`pip install viewx[pdf]`). La API ya no expone LaTeX: secciones con `with r.section(...)`, anchos como fracciones y colores CSS.


In [26]:
import shutil
if shutil.which("pdflatex"):
    Report.auto(df, title="Iris Quality Report", outdir="output").save("notebook_auto_report.pdf")
else:
    print("pdflatex no encontrado — omitiendo PDF auto")


[ViewX] ✅ PDF generado: output\notebook_auto_report.pdf


In [27]:
# Reporte manual (extracto) — sin LaTeX expuesto
r = Report(title="Reporte Manual ViewX", author="ViewX Notebook")
with r.section("Introduccion"):
    r.text("Ejemplo de reporte PDF generado con ViewX.")
    r.bullets(["Texto estructurado", "Tablas", "Graficos"])
    r.add_box("Nota", "Colores CSS en las cajas.", color="#DBEAFE")
if shutil.which("pdflatex"):
    r.save("notebook_manual_report.pdf")


[ViewX] ✅ PDF generado: output\notebook_manual_report.pdf


## 9. Flujo completo

Generar los cuatro artefactos desde el mismo dataset — una linea por artefacto.


In [28]:
import shutil
df = load_iris()
DataMatrix(df).analyze().save("output/flow_datamatrix.html")
Dashboard.auto(df).save("output/flow_dashboard.html")
Presentation.auto(df).save("output/flow_slides.html")
if shutil.which("pdflatex"):
    Report.auto(df, outdir="output").save("flow_report.pdf")
print("Artefactos generados en output/")


c:\Users\Usuario\Documents\Emmanuel Ascendra\GitHub_GhostAnalyst\Librerias_Python\Libreria_Visual\viewx\HTML\html_engine.py:1182: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



[ViewX] ✅ PDF generado: output\flow_report.pdf
Artefactos generados en output/


## Integración StatsLibX

StatsLibX analiza; ViewX visualiza al exportar. El payload `to_report_data()` se convierte con `from_report_payload()`.

Requiere: `pip install statslibx[viewx]`


In [ ]:
from statslibx import DescriptiveStats, load_iris, to_report_data
from viewx import from_report_payload

df = load_iris()
summary = DescriptiveStats(df).summary()

# Opción directa en el resultado
try:
    path = summary.to_html("viewx_from_statslibx.html", data=df, show=False)
    print("Direct export:", path)
except ImportError as exc:
    print(exc)

# Pipeline manual
payload = to_report_data(summary, include_figures=True, data=df)
from_report_payload(payload, target="html", filename="viewx_payload.html")
print("Payload export complete")

